# Cash-Secured Put Scanner

Pulls the **live put chain** for any ticker and runs your full checklist on every strike —
the same analysis you've been doing by hand in Lesson 3, done automatically.

**How to use it:**
1. Type a ticker in Step 2 (the line marked `# <- EDIT`)
2. **Runtime → Run all**
3. Read the ranked list in Step 4

**Important:** Yahoo's options data is delayed about 15 minutes and its IV can differ slightly from thinkorswim.
Use this to *find* candidates — then confirm the strike, mark and IV in thinkorswim before you place anything.

## Step 1 — Load the tools

In [ ]:
!pip install yfinance -q
import yfinance as yf
import pandas as pd
import numpy as np
from math import log, sqrt, exp, erf
from datetime import date, datetime

def N(x):
    return 0.5 * (1 + erf(x / sqrt(2)))

def put_greeks(S, K, days, iv, r=0.04):
    T = max(days, 1) / 365
    d1 = (log(S / K) + (r + iv**2 / 2) * T) / (iv * sqrt(T))
    d2 = d1 - iv * sqrt(T)
    return N(d1) - 1, N(-d2)          # delta, chance of finishing in the money

print("Tools loaded.")

## Step 2 — Your settings

In [ ]:
TICKER        = "INTC"   # <- EDIT: any optionable stock or ETF
MIN_DAYS      = 30       # <- EDIT: shortest expiration to consider
MAX_DAYS      = 45       # <- EDIT: longest expiration to consider
MIN_CHANCE    = 0.15     # your rule: at least 15% chance of assignment...
MAX_CHANCE    = 0.25     # ...and no more than 25%
MIN_ANNUAL    = 0.12     # your rule: at least 12% annualized
CASH_LIMIT    = 25_000   # <- EDIT: most cash you'd tie up in one put
MIN_OPEN_INT  = 100      # skip strikes almost nobody trades
MAX_SPREAD    = 0.15     # skip strikes where bid-ask is wider than 15% of the mid

tk = yf.Ticker(TICKER)
hist = tk.history(period="5d")
S = float(hist["Close"].iloc[-1])
print(f"{TICKER} last price: ${S:,.2f}")

earnings = None
try:
    cal = tk.calendar
    if isinstance(cal, dict):
        ed = cal.get("Earnings Date")
        earnings = ed[0] if isinstance(ed, (list, tuple)) and ed else ed
    elif cal is not None and "Earnings Date" in getattr(cal, "index", []):
        earnings = cal.loc["Earnings Date"].iloc[0]
    if earnings is not None:
        earnings = pd.Timestamp(earnings).date()
        print(f"Next earnings (per Yahoo): {earnings}")
except Exception:
    pass
if earnings is None:
    print("Couldn't find an earnings date — check it in thinkorswim before trading.")

## Step 3 — Pull the chain and score every strike

In [ ]:
today = date.today()
rows = []
for exp_str in tk.options:
    exp_date = datetime.strptime(exp_str, "%Y-%m-%d").date()
    days = (exp_date - today).days
    if days < MIN_DAYS or days > MAX_DAYS:
        continue
    puts = tk.option_chain(exp_str).puts
    for _, p in puts.iterrows():
        K, iv = float(p["strike"]), float(p.get("impliedVolatility", np.nan) or np.nan)
        bid, ask = float(p.get("bid", 0) or 0), float(p.get("ask", 0) or 0)
        if K >= S or not iv or np.isnan(iv) or iv <= 0.01:
            continue
        mid = (bid + ask) / 2 if bid > 0 and ask > 0 else float(p.get("lastPrice", 0) or 0)
        if mid <= 0:
            continue
        delta, chance = put_greeks(S, K, days, iv)
        rows.append({
            "Expiration": exp_str, "Days": days, "Strike": K, "Mid": mid,
            "IV": iv, "Delta": delta, "Chance assigned": chance,
            "Annualized": mid / K * 365 / days,
            "Buy price if assigned": K - mid,
            "Discount": 1 - (K - mid) / S,
            "Cash needed": K * 100,
            "Open interest": int(p.get("openInterest", 0) or 0),
            "Spread %": (ask - bid) / mid if bid > 0 and ask > 0 else np.nan,
            "Earnings before exp": bool(earnings and today <= earnings <= exp_date),
        })

chain = pd.DataFrame(rows)
print(f"Scored {len(chain)} out-of-the-money puts across "
      f"{chain['Expiration'].nunique() if len(chain) else 0} expirations between {MIN_DAYS} and {MAX_DAYS} days.")

## Step 4 — The candidates that pass your checklist

Every row here passes all of your rules: chance of assignment in your range, annualized return above your
minimum, cash within your limit, enough open interest, a tight enough market, and **no earnings before expiration**.
Ranked by annualized return.

In [ ]:
def show(df):
    out = df[["Expiration", "Days", "Strike", "Mid", "IV", "Delta", "Chance assigned", "Annualized",
              "Buy price if assigned", "Discount", "Cash needed", "Open interest"]].copy()
    out["Mid"] = out["Mid"].map("${:,.2f}".format)
    out["IV"] = out["IV"].map("{:.1%}".format)
    out["Delta"] = out["Delta"].map("{:+.3f}".format)
    out["Chance assigned"] = out["Chance assigned"].map("{:.1%}".format)
    out["Annualized"] = out["Annualized"].map("{:.1%}".format)
    out["Buy price if assigned"] = out["Buy price if assigned"].map("${:,.2f}".format)
    out["Discount"] = out["Discount"].map("{:.1%}".format)
    out["Cash needed"] = out["Cash needed"].map("${:,.0f}".format)
    print(out.to_string(index=False))

if chain.empty:
    print("No puts found in that expiration window. Widen MIN_DAYS / MAX_DAYS in Step 2.")
else:
    passes = chain[
        chain["Chance assigned"].between(MIN_CHANCE, MAX_CHANCE)
        & (chain["Annualized"] >= MIN_ANNUAL)
        & (chain["Cash needed"] <= CASH_LIMIT)
        & (chain["Open interest"] >= MIN_OPEN_INT)
        & ((chain["Spread %"] <= MAX_SPREAD) | chain["Spread %"].isna())
        & (~chain["Earnings before exp"])
    ].sort_values("Annualized", ascending=False)

    if passes.empty:
        print("Nothing passes every rule right now. Here's why the closest strikes failed:")
        near = chain[chain["Chance assigned"].between(MIN_CHANCE, MAX_CHANCE)]
        if near.empty:
            print("   No strikes in your 15–25% chance range.")
        else:
            print(f"   Strikes in range: {len(near)}")
            print(f"   ...with ≥{MIN_ANNUAL:.0%} annualized: {(near['Annualized'] >= MIN_ANNUAL).sum()}")
            print(f"   ...within cash limit: {(near['Cash needed'] <= CASH_LIMIT).sum()}")
            print(f"   ...with earnings before expiration: {near['Earnings before exp'].sum()}")
    else:
        print(f"{len(passes)} puts pass every rule for {TICKER} at ${S:,.2f}:\n")
        show(passes.head(10))

## Step 5 — Deep dive on one strike

Pick a strike from the list above and see the full picture — the same breakdown we've been doing together.

In [ ]:
PICK_EXPIRATION = None   # <- EDIT: e.g. "2026-12-04"  (None = use the top-ranked candidate)
PICK_STRIKE     = None   # <- EDIT: e.g. 80

src = passes if 'passes' in dir() and not passes.empty else chain
if PICK_EXPIRATION and PICK_STRIKE:
    row = chain[(chain["Expiration"] == PICK_EXPIRATION) & (chain["Strike"] == PICK_STRIKE)]
else:
    row = src.head(1)

if row.empty:
    print("That strike isn't in the scanned chain — check the expiration and strike.")
else:
    r = row.iloc[0]
    prem = r["Mid"] * 100
    be = r["Buy price if assigned"]
    print(f"{TICKER} {r['Expiration']} {r['Strike']:g} PUT  ·  {r['Days']} days  ·  IV {r['IV']:.1%}  ·  stock ${S:,.2f}")
    print()
    print(f"  Collect:               ${prem:,.0f} per contract")
    print(f"  Chance you keep it all: {1 - r['Chance assigned']:.1%}")
    print(f"  Annualized return:     {r['Annualized']:.1%}")
    print(f"  Cash to secure:        ${r['Cash needed']:,.0f}")
    print(f"  If assigned, you own at ${be:,.2f}  ({r['Discount']:.1%} below today)")
    print(f"  Close for 50% profit:  buy back at about ${r['Mid']/2:,.2f}")
    print()
    print("  At expiration:")
    for px in sorted({round(S, 2), r["Strike"], round(be, 2), round(be * 0.93, 2), round(be * 0.85, 2)}, reverse=True):
        pl = prem if px >= r["Strike"] else (px - be) * 100
        tag = "  <- breakeven" if abs(px - be) < 0.01 else ""
        sign = "" if abs(pl) < 0.5 else ("+" if pl > 0 else "−")
        print(f"     {TICKER} at ${px:>9,.2f}  →  {sign}${abs(pl):,.0f}{tag}")
    if r["Earnings before exp"]:
        print("\n  ⚠ Earnings report falls before this expiration.")

## Before you place any trade from this list

1. **Confirm in thinkorswim** — Yahoo is ~15 minutes delayed. Check the strike's current mark and Impl Vol.
2. **Confirm the earnings date** — Yahoo's calendar isn't always right.
3. **Would you be happy owning 100 shares at the "buy price if assigned"?** If not, skip it.
4. **Size:** one contract per name, and keep total cash tied up in short puts under about 40% of the account.